# Leafora — train + fine-tune on a free Colab GPU

**Everything in this notebook happens in Colab. You do not run training on your laptop.**

This notebook does four things, in order:

1. Trains the notebook's model on lab photos (stage 1) → **lab accuracy**
2. Measures that model on real field photos (before)
3. Fine-tunes it on real field photos (stage 2)
4. Measures it again on the same real photos (after) → **field accuracy**

It runs the project's own scripts. It does not reimplement the model.

---

## Before you start: three files in Google Drive

In Google Drive, make a folder named exactly **Leafora** and put these three files in it:

| File | What it holds |
|---|---|
| `leafora_backend.tgz` | the code |
| `leafora_tomato.tgz` | lab photos (PlantVillage, 10 classes) |
| `leafora_field.tgz` | real field photos (10 classes) |

If a file is missing, cell 2 stops and tells you which one.

---

## How to run this

Run the cells **top to bottom, one click each**. Ten steps. Read the line that says
`STEP OK` before moving on — that is the guard telling you the step worked.

Total time on a free T4 GPU: roughly **35–50 minutes**.

## Step 1 — Turn on the GPU

Menu: `Runtime` → `Change runtime type` → Hardware accelerator → **T4 GPU** → Save.

Then run this cell. If it prints no GPU, the runtime setting did not take — set it again.

In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
print("TensorFlow:", tf.__version__)
print("GPU devices:", gpus if gpus else "NONE")

if not gpus:
    raise SystemExit(
        "No GPU. Runtime -> Change runtime type -> T4 GPU -> Save, then run this cell again."
    )

print("STEP OK - GPU is available")

## Step 2 — Connect Drive and unpack everything

A Google pop-up asks for permission. Allow it — this is how the notebook reads the
three files from your **Leafora** folder.

The archives are unpacked into Colab's own fast disk (`/content/leafora`), not into Drive.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, subprocess

DRIVE_DIR = "/content/drive/MyDrive/Leafora"
WORK = "/content/leafora"

ARCHIVES = ["leafora_backend.tgz", "leafora_tomato.tgz", "leafora_field.tgz"]

print("Files in the Leafora Drive folder:")
present = set(os.listdir(DRIVE_DIR)) if os.path.isdir(DRIVE_DIR) else set()
for name in ARCHIVES:
    print(f"  {name:24} {'found' if name in present else 'MISSING'}")

missing = [n for n in ARCHIVES if n not in present]
if missing:
    raise SystemExit(f"Upload these to Drive/Leafora first: {missing}")

os.makedirs(WORK, exist_ok=True)
for name in ARCHIVES:
    print(f"Unpacking {name} ...")
    subprocess.run(["tar", "-xzf", os.path.join(DRIVE_DIR, name), "-C", WORK], check=True)

os.environ["DATASET_DIR"] = os.path.join(WORK, "tomato")
os.environ["FIELD_DATASET_DIR"] = os.path.join(WORK, "datasets", "tomato_field")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

print("\nSTEP OK - unpacked to", WORK)

## Step 3 — Check the photos are really there

This must show **10 class folders** for each set and a non-zero image count.
If a number is 0, the archive did not unpack — do not continue.

In [ ]:
import glob, os

WORK = "/content/leafora"
SETS = {
    "lab photos (train)": os.path.join(WORK, "tomato", "train"),
    "lab photos (val)  ": os.path.join(WORK, "tomato", "val"),
    "field photos (train)": os.path.join(WORK, "datasets", "tomato_field", "train"),
    "field photos (val)  ": os.path.join(WORK, "datasets", "tomato_field", "val"),
}

for label, path in SETS.items():
    folders = sorted(d for d in os.listdir(path)) if os.path.isdir(path) else []
    images = len(glob.glob(os.path.join(path, "*", "*")))
    print(f"{label}: {len(folders)} classes, {images} images")
    if len(folders) != 10 or images == 0:
        raise SystemExit(f"Problem with {path} - expected 10 class folders with images.")

print("\nSTEP OK - both datasets look right")

## Step 4 — Stage 1: train on lab photos

This is the notebook's own configuration, unchanged: 256×256, batch 32,
`pixel / 255.0`, frozen DenseNet121 + the notebook's head, Adam 1e-4,
categorical crossentropy, up to 100 epochs, early stopping `patience=0` on `val_loss`.

On a previous T4 run this took **12.8 minutes / 18 epochs**. Do not close the tab.

In [ ]:
%cd /content/leafora/backend
!python -m training.train

## Step 5 — Read the lab accuracy

This is the number you quote as **lab accuracy**. It is measured, not copied from
the notebook's title.

In [ ]:
import json, os

RESULTS = "/content/leafora/backend/training/results"
meta_path = os.path.join(RESULTS, "model_metadata.json")

if not os.path.exists(meta_path):
    raise SystemExit("No metadata - Stage 1 did not finish. Re-run Step 4.")

meta = json.load(open(meta_path))
m = meta["validation_metrics"]

print("Epochs completed :", meta["epochs_completed"])
print("Training time    :", round(meta["training_seconds"] / 60, 1), "minutes")
print("GPU used         :", meta["environment"]["gpu_devices"] or "CPU")
print("TensorFlow       :", meta["environment"]["tensorflow_version"])
print("Classes          :", meta["num_classes"])
print("-" * 46)
print(f"LAB ACCURACY: {m['accuracy'] * 100:.2f}%   (loss {m['loss']:.4f})")
print("STEP OK - write this number down")

## Step 6 — How good is it on real photos? (BEFORE)

Same model, scored on 1,791 real field photos it has never seen.
This is the number that matters for a phone app. Expect it to be much lower than
the lab number — that gap is the finding, not a bug.

In [ ]:
%cd /content/leafora/backend
!python -m training.evaluate_field --tag lab_model

## Step 7 — Stage 2: fine-tune on real photos

This keeps everything Stage 1 learned, unfreezes the last DenseNet block, and lets
the real photos adjust it slowly (Adam 1e-5). Settings in `training/config.py`:

| Setting | Value | Why |
|---|---|---|
| epochs | 25 | the previous 15-epoch run was still improving when it was cut off |
| patience | 8 | small noisy set — a couple of bad epochs must not end the run |
| unfreeze from | `conv5_block1` | the whole last block, not just its tail |
| field sample | 1.0 | use all of `field/train`; `field/val` still stays unseen |
| learning rate | 1e-5 | slow enough not to erase the lab features |

It prints the **before** number first, then trains, then reloads the **best** checkpoint
(not the last epoch), so the exported file is the strongest one.

In [ ]:
%cd /content/leafora/backend
!python -m training.finetune_field

## Step 8 — The AFTER number

Same 1,791 photos, same script, only the weights changed. The difference between
this and Step 6 is the real, honest measure of whether fine-tuning helped.

In [ ]:
%cd /content/leafora/backend
!python -m training.evaluate_field --model models/exported/leafora_densenet121_field.keras --tag field_model

## Step 9 — Save everything back to Drive

Colab deletes its disk when the session ends. This cell packs the models and every
result into one file on Drive: **`leafora_results.tgz`**.

Run it as soon as Step 8 finishes.

In [ ]:
import os, shutil, subprocess

DRIVE_DIR = "/content/drive/MyDrive/Leafora"
BACKEND = "/content/leafora/backend"
OUT = "/content/leafora_out"

shutil.rmtree(OUT, ignore_errors=True)
os.makedirs(OUT, exist_ok=True)

for folder in ("models", "training/results", "training/checkpoints"):
    src = os.path.join(BACKEND, folder)
    dst = os.path.join(OUT, folder)
    if os.path.isdir(src):
        shutil.copytree(src, dst)
        print("collected", folder)
    else:
        print("nothing in", folder)

class_names = os.path.join(BACKEND, "app", "ml", "class_names.json")
if os.path.exists(class_names):
    os.makedirs(os.path.join(OUT, "app", "ml"), exist_ok=True)
    shutil.copy(class_names, os.path.join(OUT, "app", "ml", "class_names.json"))
    print("collected class_names.json")

archive = os.path.join(DRIVE_DIR, "leafora_results.tgz")
subprocess.run(["tar", "-czf", archive, "-C", OUT, "."], check=True)

size = os.path.getsize(archive) / 1e6
print(f"\nSTEP OK - saved Drive/Leafora/leafora_results.tgz ({size:.1f} MB)")

## Step 10 — Copy the numbers down

This prints the three numbers you need for the write-up. If you only remember one
thing: **lab accuracy and field accuracy are two different results**, and both are real.

In [ ]:
import json, os

RESULTS = "/content/leafora/backend/training/results"

def load(name):
    path = os.path.join(RESULTS, name)
    return json.load(open(path)) if os.path.exists(path) else None

lab = load("model_metadata.json")
before = load("field_evaluation_lab_model.json")
after = load("field_evaluation_field_model.json")
finetune = load("finetune_metadata.json")

rows = []
if lab:
    rows.append(("Lab accuracy (stage 1)", f"{lab['validation_metrics']['accuracy'] * 100:.2f}%"))
if before:
    rows.append(("Field accuracy, before", f"{before['metrics']['accuracy'] * 100:.2f}%"))
if after:
    rows.append(("Field accuracy, after", f"{after['metrics']['accuracy'] * 100:.2f}%"))
if before and after:
    gain = (after['metrics']['accuracy'] - before['metrics']['accuracy']) * 100
    rows.append(("Improvement", f"{gain:+.2f} points"))

print("=" * 52)
print("LEAFORA - MEASURED ON A COLAB T4 GPU")
print("=" * 52)
for label, value in rows:
    print(f"{label:26} {value}")
print("=" * 52)

if lab:
    print(f"Trained on {lab['train_images']} lab images, validated on {lab['val_images']},"
          f" {lab['epochs_completed']} epochs.")
if after:
    print(f"Field result measured on {after['field_validation_images']} real photos.")
if finetune:
    print(f"Fine-tune ran {finetune['epochs_completed']} epochs"
          f" (unfrozen from {finetune['unfreeze_from']}, lr {finetune['learning_rate']}).")
print("\nSTEP OK - notebook finished")

---

## What to expect, and what to do if it is lower than you hoped

Realistic result from the previous run on this project:

| Number | Value |
|---|---|
| Lab accuracy | 93.10% |
| Field accuracy, before fine-tuning | 59.24% |
| Field accuracy, after fine-tuning | 66.22% (with the old, weaker settings) |

This run uses more epochs, the whole field training split and the best checkpoint, so
**the after number should be higher than 66.22%**. It will not reach the lab number —
real photos are genuinely harder.

If the after number is *not* higher, do not hide it. Report both numbers and say the
fine-tune did not help. A negative result measured properly is still a result.

---

## Writing it up honestly

Correct:

> The reproduced model reached **93.10%** validation accuracy on lab-style images
> (10,000 training images, 1,000 validation, 18 epochs, NVIDIA T4). On 1,791 real field
> photos the same model scored **59.24%**. After a fine-tuning stage on field data it
> scored **X%** on those same photos.

Do not write:

- "Leafora achieves 94% accuracy" — that is the notebook's title, not a measurement.
- "The model works on real photos" without the field number.
- "Sensor readings improve detection" — no multimodal model was trained.

Also state that training ran on a **Colab T4**, not on the laptop's RTX 5050.

---

## If something breaks

| What you see | What to do |
|---|---|
| `No GPU` in Step 1 | Runtime → Change runtime type → T4 GPU → Save, re-run Step 1 |
| `Upload these to Drive/Leafora first` | the named file is missing from the Drive folder |
| `Problem with .../train` in Step 3 | that archive did not unpack — re-run Step 2 from a fresh runtime |
| Colab disconnected | Runtime → Reconnect, then run Steps 2–4 again; a lost session keeps nothing |
| `No metadata` in Step 5 | Step 4 crashed — scroll up to its error before continuing |
| Fine-tune took the whole session | it stops by itself at 25 epochs or when it stalls; check Drive for `leafora_results.tgz` first |

Free Colab has a time limit. If Stage 2 gets cut off, Stage 1's model and the before
number are still valid on their own — run Step 9 anyway.